# kagome-vqe demo

A thin walk-through of the package API. Everything here is also available from the `kagome-vqe` command line; the notebook exists so you can poke at intermediate objects.

Run `uv sync --all-groups` first and select the `.venv` kernel.

In [ ]:
import numpy as np

from kagome_vqe import exact_spectrum, heisenberg_hamiltonian, kagome_unit_cell
from kagome_vqe.ansatz import HVAnsatz, RVBAnsatz

lattice = kagome_unit_cell()
hamiltonian = heisenberg_hamiltonian(lattice)
print(lattice.num_sites, "sites,", lattice.num_bonds, "bonds,", len(hamiltonian), "Pauli terms")
print("site order:", lattice.site_order, "ring bond fraction:", lattice.ring_bond_fraction())
print("lowest eigenvalues:", exact_spectrum(hamiltonian, k=3))

## Ansatzes

Both ring ansatzes start from six singlets. With the default Hamiltonian-cycle site order the all-zero parameter point is already the exact ground state.

In [ ]:
from qiskit.quantum_info import Statevector

for ansatz in (RVBAnsatz(), HVAnsatz()):
    circuit = ansatz.build()
    energy = Statevector(circuit.assign_parameters(np.zeros(ansatz.num_parameters))).expectation_value(hamiltonian).real
    print(f"{ansatz.name}: {ansatz.num_parameters} parameters, E(0) = {energy:.6f}")

HVAnsatz().build().draw("mpl", fold=-1)

## A run from a config

`RunConfig` is the same object the CLI builds from TOML. Overrides here are plain keyword arguments.

In [ ]:
from kagome_vqe.config import BackendConfig, OptimizerConfig, RunConfig
from kagome_vqe.vqe import prepare, run_vqe

cfg = RunConfig(
    name="demo",
    seed=7,
    optimizer=OptimizerConfig(type="spsa", maxiter=40),
    backend=BackendConfig(kind="statevector"),
)
prepared = prepare(cfg)
prepared.summary()

In [ ]:
result = run_vqe(cfg, prepared=prepared)
print(f"E = {result.energy:.6f}  (exact {result.exact_energy:.6f}, rel. err {result.relative_error:.2e})")

from kagome_vqe.plotting import plot_convergence

plot_convergence({"hva + spsa": result.energies_per_iteration()}, result.exact_energy);

## Noisy fake backend

Swap the backend and nothing else changes. `aer_noisy` imports the fake device's calibration data into Aer; the transpiled circuit and the laid-out observable are visible on `prepared.problem`.

In [ ]:
from kagome_vqe.lattice import GUADALUPE_HEAVY_HEX_LOOP

noisy = cfg.model_copy(
    update={
        "name": "demo_noisy",
        "optimizer": OptimizerConfig(type="spsa", maxiter=3, lr=0.05, perturbation=0.1),
        "backend": BackendConfig(
            kind="aer_noisy", name="FakeGuadalupeV2", shots=1024, initial_layout=list(GUADALUPE_HEAVY_HEX_LOOP)
        ),
    }
)
noisy_prepared = prepare(noisy)
print(noisy_prepared.problem.stats())
noisy_result = run_vqe(noisy, prepared=noisy_prepared)
print(f"noisy E = {noisy_result.best_energy:.4f}")